# Pilot V4 - Literature-Based Telecom Tower Class Library

This notebook does not calculate collapse probability. Instead, it builds the class vocabulary needed before a Wang-style class-based fragility framework.

Kid version: before making fragility curves for many tower types, we first make a clean dictionary of tower types from the literature. We do not invent new types or fake numbers.

## 1. Imports and Paths

This notebook creates simple tables and saves them as CSV/JSON outputs.

In [ ]:
from pathlib import Path
import json

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'notebook_v4'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')
print(f'Notebook outputs: {OUTPUT_DIR}')

## 2. Literature Sources

This class library is based on the telecom portfolio fragility work associated with Khazaali and Bocchini. The important idea used here is the class list: water tank, monopole, guyed, and lattice tower.

The notebook intentionally stops at class labels and the wind-speed convention. It does not invent class-specific fragility parameters.

In [ ]:
class_library_metadata = {
    'project_name': 'Notebook Pilot V4 literature-grounded telecom tower class library',
    'purpose': 'Create a reusable telecom tower class catalog using labels explicitly listed in portfolio fragility literature.',
    'primary_sources': [
        {
            'title': 'Physics-Based and Data-Driven Portfolio Fragility Curves for Telecommunication Towers Under Hurricanes',
            'authors': ['Mohanad Khazaali', 'Paolo Bocchini'],
            'year': 2022,
            'source_type': 'official research group summary',
            'url': 'https://www.lehigh.edu/~pab409/20220607emi.html',
            'key_points_used': [
                'Recorded wind speeds were converted to 2 min sustained wind at 10 m height.',
                'Structural classes include water tank, monopole, guyed, and lattice tower.',
            ],
        },
        {
            'title': 'Damage and Resilience Assessments of Telecommunication Systems under Hurricanes',
            'authors': ['Mohanad Khazaali'],
            'year': 2022,
            'source_type': 'dissertation summary',
            'url': 'https://preserve.lehigh.edu/lehigh-scholarship/graduate-publications-theses-dissertations/theses-dissertations/damage',
            'key_points_used': [
                'The structural collapse analysis uses collapse fragility curves for different structural classes.',
                'The listed telecom classes include water tank, monopole, guyed, and lattice towers.',
            ],
        },
    ],
}

intensity_measure_standard = {
    'name': '2 min sustained wind speed at 10 m height',
    'hazard_type': 'wind',
    'units': 'm/s',
    'source_basis': 'Khazaali and Bocchini telecom portfolio fragility summary',
}

class_library_metadata

## 3. Build the Class Library Table

This table preserves the class labels from the literature. It is the beginning of the class-based framework.

In [ ]:
literature_class_library = [
    {
        'class_id': 'water_tank',
        'display_label': 'water tank',
        'source_label': 'water tank',
        'category_type': 'telecommunication tower structural class',
        'source_status': 'explicitly listed in telecom portfolio fragility summary',
    },
    {
        'class_id': 'monopole',
        'display_label': 'monopole',
        'source_label': 'monopole',
        'category_type': 'telecommunication tower structural class',
        'source_status': 'explicitly listed in telecom portfolio fragility summary',
    },
    {
        'class_id': 'guyed_tower',
        'display_label': 'guyed tower',
        'source_label': 'guyed',
        'category_type': 'telecommunication tower structural class',
        'source_status': 'explicitly listed in telecom portfolio fragility summary',
    },
    {
        'class_id': 'lattice_tower',
        'display_label': 'lattice tower',
        'source_label': 'lattice tower',
        'category_type': 'telecommunication tower structural class',
        'source_status': 'explicitly listed in telecom portfolio fragility summary',
    },
]

class_library_df = pd.DataFrame(literature_class_library)
class_library_df

## 4. Validate and Normalize Class Labels

Real datasets often use slightly different labels. This helper converts obvious labels into the clean class vocabulary and rejects unsupported labels instead of guessing.

In [ ]:
def normalize_literature_class_label(raw_label):
    '''Normalize a raw telecom tower class label using only the literature-backed vocabulary.'''
    if raw_label is None:
        raise ValueError('raw_label cannot be None')

    cleaned = str(raw_label).strip().lower().replace('-', ' ')
    cleaned = ' '.join(cleaned.split())

    exact_mapping = {
        'water tank': 'water tank',
        'monopole': 'monopole',
        'guyed': 'guyed tower',
        'guyed tower': 'guyed tower',
        'lattice': 'lattice tower',
        'lattice tower': 'lattice tower',
        'lattice towers': 'lattice tower',
    }

    if cleaned not in exact_mapping:
        raise ValueError(f'Unsupported class label: {raw_label}')

    return exact_mapping[cleaned]


def validate_inventory_classes(inventory_df):
    '''Check that an inventory has required columns and normalize its class labels.'''
    required_columns = ['tower_id', 'reported_class_label']
    missing_columns = [column for column in required_columns if column not in inventory_df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    validated_df = inventory_df.copy()
    validated_df['normalized_class_label'] = validated_df['reported_class_label'].apply(normalize_literature_class_label)
    return validated_df


example_inventory_df = pd.DataFrame([
    {'tower_id': 'EX-001', 'reported_class_label': 'water tank'},
    {'tower_id': 'EX-002', 'reported_class_label': 'monopole'},
    {'tower_id': 'EX-003', 'reported_class_label': 'guyed'},
    {'tower_id': 'EX-004', 'reported_class_label': 'lattice tower'},
])

validated_inventory_df = validate_inventory_classes(example_inventory_df)
validated_inventory_df

## 5. Simple Summary Table

This is the clean table you can show as the starting point for a class-based telecom fragility framework.

In [ ]:
class_summary_df = class_library_df[
    ['class_id', 'display_label', 'source_label', 'category_type', 'source_status']
].copy()

class_summary_df

## 6. Save Notebook Outputs

The saved outputs are small but important: class metadata, intensity-measure convention, class library, summary table, and an example inventory template.

In [ ]:
with (OUTPUT_DIR / 'class_library_metadata.json').open('w', encoding='utf-8') as file:
    json.dump(class_library_metadata, file, indent=4)

with (OUTPUT_DIR / 'intensity_measure_standard.json').open('w', encoding='utf-8') as file:
    json.dump(intensity_measure_standard, file, indent=4)

class_library_df.to_csv(OUTPUT_DIR / 'literature_class_library.csv', index=False)
class_summary_df.to_csv(OUTPUT_DIR / 'literature_class_summary.csv', index=False)
validated_inventory_df.to_csv(OUTPUT_DIR / 'example_inventory_template.csv', index=False)

print(f'Saved notebook V4 outputs to: {OUTPUT_DIR}')
print('Class labels preserved:')
for label in class_summary_df['display_label']:
    print(f'- {label}')